# Hướng dẫn chạy DuplexChat trên Kaggle
Notebook này thiết lập môi trường hoàn chỉnh để chạy pipeline tách nguồn âm thanh (Source Separation) trực tiếp trên nền tảng Kaggle.

### 1. Chuẩn bị môi trường (Cài đặt ffmpeg & uv)

In [ ]:
!apt-get update && apt-get install -y ffmpeg
!pip install uv

### 2. Chuẩn bị source từ zip đã upload
Upload `duplexchat_project.zip` vào Kaggle input, sau đó giải nén vào `/kaggle/working/duplexchat_project`.

In [ ]:
from pathlib import Path

ZIP_PATH = Path("/kaggle/input/duplexchat-project/duplexchat_project.zip")
PROJECT_ROOT = Path("/kaggle/working/duplexchat_project")
!rm -rf "$PROJECT_ROOT"
!mkdir -p "$PROJECT_ROOT"
!unzip -q "$ZIP_PATH" -d "$PROJECT_ROOT"
%cd /kaggle/working/duplexchat_project

### 3. Cài đặt các thư viện (Dependencies)
Đồng bộ toàn bộ các packages thông qua `uv`.

In [ ]:
!uv sync

### 4. Xác thực HuggingFace Token
Bạn cần thêm `HF_TOKEN` vào **Kaggle Secrets** (Menu Add-ons -> Secrets) để quyền tải 2 model Diarization và Separation.

In [ ]:
from kaggle_secrets import UserSecretsClient
import os

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
    !hf auth login --token $HUGGING_FACE_HUB_TOKEN
except Exception as e:
    print("Lỗi: Không tìm thấy HF_TOKEN trong Kaggle Secrets. Vui lòng vào Add-ons -> Secrets để thêm bí mật có tên 'HF_TOKEN'.")

### 5. Chạy end-to-end test theo model
Chọn model trong `ENABLED_TEST_NAMES`; mỗi lần chạy lưu audio vào thư mục riêng theo đúng model để nghe lại.

In [ ]:
!uv add wrapt

In [ ]:
!uv run python -c "import wrapt, numpy; print('wrapt OK'); print(numpy.__version__)"

In [ ]:
from pathlib import Path

AUDIO_PATH = "/kaggle/input/datasets/ngocbaotrinhtuan/inputs/real.wav"
OUT_ROOT = Path("/kaggle/working/model_audio_tests")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_TESTS = [
    {
        "name": "pyannote31__dialoguesidon",
        "install": "uv sync --extra diarization-pyannote --extra separation-dialoguesidon",
        "diarization_backend": "pyannote",
        "diarization_model": "pyannote/speaker-diarization-3.1",
        "separation_backend": "dialoguesidon",
        "separation_model": "sarulab-speech/DialogueSidon",
    },
    {
        "name": "sortformer__dialoguesidon",
        "install": "uv sync --extra diarization-sortformer --extra separation-dialoguesidon",
        "diarization_backend": "sortformer",
        "diarization_model": "nvidia/diar_sortformer_4spk-v1",
        "separation_backend": "dialoguesidon",
        "separation_model": "sarulab-speech/DialogueSidon",
    },
    {
        "name": "diarizen__dialoguesidon",
        "install": "uv pip install -r requirements/diarization-diarizen.txt && uv sync --extra separation-dialoguesidon",
        "diarization_backend": "diarizen",
        "diarization_model": "BUT-FIT/diarizen-wavlm-large-s80-md",
        "separation_backend": "dialoguesidon",
        "separation_model": "sarulab-speech/DialogueSidon",
    },
    {
        "name": "pyannote31__sepformer",
        "install": "uv sync --extra diarization-pyannote --extra separation-sepformer",
        "diarization_backend": "pyannote",
        "diarization_model": "pyannote/speaker-diarization-3.1",
        "separation_backend": "sepformer",
        "separation_model": "speechbrain/sepformer-wsj02mix",
    },
    {
        "name": "pyannote31__mossformer2",
        "install": "uv pip install -r requirements/separation-mossformer2.txt && uv sync --extra diarization-pyannote",
        "diarization_backend": "pyannote",
        "diarization_model": "pyannote/speaker-diarization-3.1",
        "separation_backend": "mossformer2",
        "separation_model": "alibabasglab/MossFormer2_SS_16K",
    },
]

RUN_ALL_MODELS = True
ENABLED_TEST_NAMES = [test["name"] for test in MODEL_TESTS] if RUN_ALL_MODELS else ["pyannote31__dialoguesidon"]
FAILED_TESTS = []

for test in MODEL_TESTS:
    if test["name"] not in ENABLED_TEST_NAMES:
        continue
    out_dir = OUT_ROOT / test["name"]
    out_dir.mkdir(parents=True, exist_ok=True)
    output_prefix = out_dir / "speaker"
    print(f"\n=== Running {test['name']} ===")
    print(f"Audio will be saved under: {out_dir}")
    install_cmd = test.get("install")
    if install_cmd:
        !{install_cmd}
    output_dir = out_dir / "outputs"
    !MPLBACKEND=Agg uv run python test_single.py "$AUDIO_PATH"         --diarize-chunk 240         --separate-chunk 240         --diarization-backend "{test['diarization_backend']}"         --diarization-model "{test['diarization_model']}"         --separation-backend "{test['separation_backend']}"         --separation-model "{test['separation_model']}"         --output-prefix "$output_prefix"         --output-dir "$output_dir"

# Hiển thị nhanh các test lỗi nếu bạn thêm try/except quanh command phía trên.
print("FAILED_TESTS:", FAILED_TESTS)


### 6. Nghe lại kết quả

In [ ]:
from pathlib import Path
import IPython.display as ipd

AUDIO_PATH = "/kaggle/input/datasets/ngocbaotrinhtuan/inputs/real.wav"
OUT_ROOT = Path("/kaggle/working/model_audio_tests")

print("Audio gốc:")
display(ipd.Audio(AUDIO_PATH))

for out_dir in sorted(OUT_ROOT.glob("*")):
    spk_a = out_dir / "speaker_A.wav"
    spk_b = out_dir / "speaker_B.wav"
    if not spk_a.exists() or not spk_b.exists():
        continue
    print(f"\nModel test: {out_dir.name}")
    print("Giọng Người A:")
    display(ipd.Audio(str(spk_a)))
    print("Giọng Người B:")
    display(ipd.Audio(str(spk_b)))
